In [1]:
from edgar import set_identity, Company, get_filings
from edgar.entity import EntityFiling, EntityFilings
from edgar import get_current_filings
from pprint import pprint
import json


set_identity("Loy Yeeko koloyyee@gmail.com")

/Users/loyyeeko/Code/Personal/Projects/insider-daily/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
hood = Company("HOOD")
hood.get_filings(form="4").latest()


╭────────────── Form 4 Robinhood Markets, Inc. [1783879]  ──────────────╮
│                                                                       │
│   Accession Number       Filing Date   Period of Report   Documents   │
│  ───────────────────────────────────────────────────────────────────  │
│   0002108589-26-000016   2026-06-15    2026-06-15         1           │
│                                                                       │
╰─ Statement of changes in beneficial ownership • filing.docs for usage─╯

In [ ]:
rddt = Company("RDDT")
last_5_form4 = rddt.get_filings(form = "4").latest(5)

Filing(company='Reddit, Inc.', cik=1713445, form='4', filing_date='2026-06-10', accession_no='0001713445-26-000086')
Filing(company='Reddit, Inc.', cik=1713445, form='4', filing_date='2026-06-10', accession_no='0001713445-26-000085')
Filing(company='Reddit, Inc.', cik=1713445, form='4', filing_date='2026-06-10', accession_no='0001713445-26-000084')
Filing(company='Reddit, Inc.', cik=1713445, form='4', filing_date='2026-06-10', accession_no='0001713445-26-000083')
Filing(company='Reddit, Inc.', cik=1713445, form='4', filing_date='2026-06-10', accession_no='0001713445-26-000082')


In [5]:
from edgar import get_filings

filings = get_filings(form=4, year=2026, quarter=[1,2] )
for f in filings[:5]:
    try:
        form4 = f.obj()
        if form4:
            summary = form4.get_ownership_summary()
            print(summary)
            if "Purchase" in summary.primary_activity and summary.net_change > 10000:
                print(f"{summary.insider_name} bought {summary.net_change:,} shares of {summary.issuer}")
            #elif "Sale" in summary.primary_activity and  summary.net_change < -10000:
            #    print(f"{summary.insider_name} sold {summary.net_change:,} shares of {summary.issuer}")
    except:
        print("something wrong.")

TransactionSummary(reporting_date='2026-06-11', issuer_name='111, Inc.', issuer_ticker='YI', insider_name='Jian David Sun', position='Director', form_type='4', remarks='', transactions=[TransactionActivity(transaction_type='sale', code='S', shares=29280, value=7905.6, price_per_share=0.27, description='', security_type='non-derivative', security_title='RSUs (Class A) [F4]', underlying_security='', exercise_date=None, expiration_date=None, footnote_ids='', footnotes_text=''), TransactionActivity(transaction_type='sale', code='S', shares=70440, value=17610.0, price_per_share=0.25, description='', security_type='non-derivative', security_title='RSUs (Class A) [F5]', underlying_security='', exercise_date=None, expiration_date=None, footnote_ids='', footnotes_text=''), TransactionActivity(transaction_type='sale', code='S', shares=12000, value=2760.0, price_per_share=0.23, description='', security_type='non-derivative', security_title='RSUs (Class A)', underlying_security='', exercise_date=N

In [3]:
import feedparser

form4_rss = "https://www.sec.gov/cgi-bin/browse-edgar?action=getcurrent&CIK=&type=4&company=&dateb=&owner=only&start=0&count=200&output=atom"
feedparser.USER_AGENT = "dko@gmail.com"

In [4]:
import re
feed = feedparser.parse(form4_rss)

#for entry in feed.entries:
    #print(entry.title, entry.link, entry.updated)

def filing_info(entry) :
    date_pattern = r"\b[0-9]{4}-[0-9]{2}-[0-9]{2}\b"
    acc_no_pattern = r"\b[0-9]{10}-[0-9]{2}-[0-9]{6}\b"
    date = re.findall(date_pattern, entry.summary)[0] if re.findall(date_pattern, entry.summary) else "" 
    acc_no = re.findall(acc_no_pattern, entry.summary)[0] if re.findall(acc_no_pattern, entry.summary) else ""
    title = entry.title
    opening = title.find("(")
    closing = title.find(")")
    cik = title[opening + 1:closing]
    return {"date": date, "cik": int(cik), "acc_no": acc_no, "title": title }

filing_basis = list(filter( lambda f: "Reporting" not in f["title"],map( filing_info, feed.entries) ))
print(filing_basis)

[{'date': '2026-06-16', 'cik': 1883814, 'acc_no': '0001965927-26-000003', 'title': '4 - Southland Holdings, Inc. (0001883814) (Issuer)'}, {'date': '2026-06-16', 'cik': 1015739, 'acc_no': '0001193125-26-273111', 'title': '4 - AWARE INC /MA/ (0001015739) (Issuer)'}, {'date': '2026-06-16', 'cik': 1528396, 'acc_no': '0001784996-26-000058', 'title': '4 - Guidewire Software, Inc. (0001528396) (Issuer)'}, {'date': '2026-06-16', 'cik': 1528396, 'acc_no': '0001805719-26-000014', 'title': '4 - Guidewire Software, Inc. (0001528396) (Issuer)'}, {'date': '2026-06-16', 'cik': 1528396, 'acc_no': '0001496152-26-000009', 'title': '4 - Guidewire Software, Inc. (0001528396) (Issuer)'}, {'date': '2026-06-16', 'cik': 1528396, 'acc_no': '0001586253-26-000008', 'title': '4 - Guidewire Software, Inc. (0001528396) (Issuer)'}, {'date': '2026-06-16', 'cik': 1528396, 'acc_no': '0001917389-26-000008', 'title': '4 - Guidewire Software, Inc. (0001528396) (Issuer)'}, {'date': '2026-06-16', 'cik': 1883814, 'acc_no': '

In [126]:
from edgar import Filing

def get_summary(comp_entry):
    filing = Filing(company=str(comp_entry["cik"]), cik=comp_entry["cik"], form="4", 
                    filing_date=comp_entry["date"], accession_no=comp_entry["acc_no"])
    f4 = filing.obj()
    #print(f4.insider_name)
    ## The meat!
    #print(f4.get_ownership_summary())
    #print(filing.filing_url)
    return {"transaction_summary": f4.get_ownership_summary(), "filing_url": filing.filing_url}
    
summaries = list(map(get_summary, filing_basis))
print(summaries)
    
    #company = filing.get_entity()
    #f4s = company.get_filings(form="4")
    #for f4 in f4s:
    #    print(f4.primary_documents)
## or more practically, search by CIK
#from edgar import get_filings

#filings = get_filings(form="4", cik=1530721)
#for filing in filings:
#    print(filing)


[{'transaction_summary': TransactionSummary(reporting_date='2026-06-05', issuer_name='UNITED FIRE GROUP INC', issuer_ticker='UFCS', insider_name='George D Milligan', position='Director', form_type='4', remarks='', transactions=[TransactionActivity(transaction_type='purchase', code='P', shares=4500.0, value=203535.0, price_per_share=45.23, description='', security_type='non-derivative', security_title='Common Stock', underlying_security='', exercise_date=None, expiration_date=None, footnote_ids='', footnotes_text='')], remaining_shares=np.float64(90033.5393), has_derivative_transactions=False), 'filing_url': 'https://www.sec.gov/Archives/edgar/data/101199/000102371026000003/primarydocument.xml'}, {'transaction_summary': TransactionSummary(reporting_date='2026-06-09', issuer_name='Webull Corp', issuer_ticker='BULL', insider_name='Walter A. Bishop', position='Director', form_type='4', remarks='', transactions=[TransactionActivity(transaction_type='exercise', code='M', shares=12500, value=

In [111]:
from edgar import get_all_current_filings, get_filings
from edgar.enums import FormType

#filings = get_filings(form="4", filing_date="2026-06-08")
filings = get_current_filings(form="4", owner = 'include', )
#filings = get_all_current_filings(form="4" )

for filing in filings:
    f4 = filing.obj()
    print(type(f4) is FormType.PROSPECTUS_424B1)
    if type(f4) is FormType.PROSPECTUS_424B1: continue

    #print(f4.insider_name)
    ## The meat!
    #print(f4.get_ownership_summary())
    #print(filing.filing_url)
    

False
False
False
False
False
False
False


No XBRL attachments found in filing Filing(form='497J', filing_date='2026-06-09', company='FT 13013', cik=2121128, accession_no='0001445546-26-004274')
No XBRL attachments found in filing Filing(form='497J', filing_date='2026-06-09', company='FT 12999', cik=2119999, accession_no='0001445546-26-004273')
No XBRL attachments found in filing Filing(form='497J', filing_date='2026-06-09', company='FT 12972', cik=2112775, accession_no='0001445546-26-004272')


False
False
False
False
False
False
False


No XBRL attachments found in filing Filing(form='497J', filing_date='2026-06-09', company='FT 12971', cik=2112774, accession_no='0001445546-26-004271')


False
False


No XBRL attachments found in filing Filing(form='485BXT', filing_date='2026-06-09', company='Investment Managers Series Trust II', cik=1587982, accession_no='0001213900-26-066562')
No XBRL attachments found in filing Filing(form='485BXT', filing_date='2026-06-09', company='Investment Managers Series Trust II', cik=1587982, accession_no='0001213900-26-066562')
No XBRL attachments found in filing Filing(form='485BXT', filing_date='2026-06-09', company='Investment Managers Series Trust II', cik=1587982, accession_no='0001213900-26-066561')
No XBRL attachments found in filing Filing(form='485BXT', filing_date='2026-06-09', company='Investment Managers Series Trust II', cik=1587982, accession_no='0001213900-26-066561')


False
False
False
False
False
False
False
False
False
False
False
False
False
False
False


No XBRL attachments found in filing Filing(form='425', filing_date='2026-06-09', company='Axiom Intelligence Acquisition Corp 1', cik=2057030, accession_no='0001213900-26-066554')


False
False
False
False
False
False
False
False
False
